# SSL con Certbot

Certificados HTTPS gratuitos con Let's Encrypt

## Introducción

HTTPS ya no es opcional: los navegadores marcan los sitios HTTP como "No seguro", Google penaliza en SEO y las APIs modernas lo exigen. Let's Encrypt ofrece certificados SSL gratuitos y automáticos. Certbot los instala y renueva por ti. En esta lección aprenderás todo el proceso.

### Objetivos de Aprendizaje

- Instalar Certbot usando snap en Ubuntu/Debian
- Obtener certificados SSL gratuitos para tu dominio
- Configurar renovación automática con cron o systemd
- Entender la configuración SSL de nginx generada por Certbot
- Crear certificados autofirmados para entornos locales/staging
- Diagnosticar problemas comunes de certificados

## Instalar Certbot con Snap

> Snap es el método recomendado por Certbot para instalarlo. Garantiza que siempre tengas la versión más reciente con actualizaciones automáticas. El proceso tiene tres pasos: remover versiones anteriores, instalar snap core, instalar certbot.

In [ ]:
instalacion_certbot = """
# 1. Eliminar versiones antiguas de certbot (si existen)
apt remove certbot

# 2. Instalar snap core y mantenerlo actualizado
snap install core
snap refresh core

# 3. Instalar certbot via snap
snap install --classic certbot

# 4. Crear symlink para usar 'certbot' como comando global
ln -s /snap/bin/certbot /usr/bin/certbot

# 5. Verificar instalación
certbot --version
"""

print(instalacion_certbot)

prerequisitos = {
    "Nginx instalado": "nginx -v",
    "Puerto 80 abierto": "ufw status | grep 80",
    "Puerto 443 abierto": "ufw status | grep 443",
    "Dominio apunta al servidor": "dig +short tudominio.com",
    "nginx.conf con server_name": "grep server_name /etc/nginx/nginx.conf",
}

print("Prerequisitos antes de ejecutar certbot:")
for requisito, comando in prerequisitos.items():
    print(f"  ✓ {requisito}")

## Obtener Certificado SSL

> Certbot puede configurar nginx automáticamente (modo --nginx) o solo obtener el certificado sin tocar nginx (modo certonly). El modo --nginx es más conveniente para la mayoría de casos: detecta tu configuración de nginx y la modifica para HTTPS.

In [ ]:
obtener_certificado = """
# Modo automático: Certbot modifica nginx automáticamente
certbot --nginx -d tudominio.com -d www.tudominio.com

# Para múltiples dominios
certbot --nginx \\
    -d tudominio.com \\
    -d www.tudominio.com \\
    -d api.tudominio.com

# Modo manual: solo obtiene el certificado
certbot certonly --nginx -d tudominio.com

# Modo standalone (sin nginx)
certbot certonly --standalone -d tudominio.com
"""

print(obtener_certificado)

archivos_certificado = {
    "/etc/letsencrypt/live/tudominio.com/cert.pem": "Certificado del dominio",
    "/etc/letsencrypt/live/tudominio.com/chain.pem": "Cadena de certificación",
    "/etc/letsencrypt/live/tudominio.com/fullchain.pem": "Cert + cadena (usar en nginx)",
    "/etc/letsencrypt/live/tudominio.com/privkey.pem": "Clave privada (mantener segura!)",
}

print("\nArchivos generados:")
for ruta, descripcion in archivos_certificado.items():
    print(f"  {ruta}")
    print(f"    → {descripcion}")

## Renovación Automática

> Los certificados de Let's Encrypt duran 90 días. Certbot instala automáticamente un timer de systemd que intenta renovar dos veces al día. Si el certificado tiene menos de 30 días para expirar, lo renueva. Puedes probarlo con --dry-run.

In [ ]:
import datetime

renovacion = """
# Probar renovación sin hacer cambios reales
certbot renew --dry-run

# Timer systemd de Certbot
systemctl status snap.certbot.renew.timer

# Ver cuando vencen tus certificados
certbot certificates

# Forzar renovación
certbot renew --force-renewal
"""

print(renovacion)

def verificar_vencimiento(dominio, dias_alerta=30):
    print(f"Verificando certificado de: {dominio}")
    fecha_vencimiento_simulada = datetime.date.today() + datetime.timedelta(days=45)
    dias_restantes = (fecha_vencimiento_simulada - datetime.date.today()).days
    
    if dias_restantes < dias_alerta:
        estado = f"ALERTA: Vence en {dias_restantes} días"
    else:
        estado = f"OK: Vence en {dias_restantes} días"
    
    print(f"  Fecha de vencimiento: {fecha_vencimiento_simulada}")
    print(f"  Estado: {estado}")
    return dias_restantes

verificar_vencimiento("miapp.com")

## Configuración SSL de Nginx

> Cuando Certbot modifica nginx, agrega la configuración SSL automáticamente. Es importante entender qué hace cada directiva para poder ajustar la seguridad según tus necesidades.

In [ ]:
config_ssl_nginx = """
server {
    listen 80;
    server_name tudominio.com www.tudominio.com;
    return 301 https://$host$request_uri;
}

server {
    listen 443 ssl;
    server_name tudominio.com www.tudominio.com;
    
    ssl_certificate /etc/letsencrypt/live/tudominio.com/fullchain.pem;
    ssl_certificate_key /etc/letsencrypt/live/tudominio.com/privkey.pem;
    
    include /etc/letsencrypt/options-ssl-nginx.conf;
    ssl_dhparam /etc/letsencrypt/ssl-dhparams.pem;
    
    add_header Strict-Transport-Security "max-age=31536000; includeSubDomains" always;
    
    ssl_stapling on;
    ssl_stapling_verify on;
    
    location / {
        proxy_pass http://localhost:8000;
        proxy_set_header Host $host;
        proxy_set_header X-Real-IP $remote_addr;
        proxy_set_header X-Forwarded-For $proxy_add_x_forwarded_for;
        proxy_set_header X-Forwarded-Proto $scheme;
    }
}
"""

print(config_ssl_nginx)

headers_seguridad = {
    "Strict-Transport-Security": "max-age=31536000; includeSubDomains",
    "X-Content-Type-Options": "nosniff",
    "X-Frame-Options": "DENY",
    "X-XSS-Protection": "1; mode=block",
}

print("Headers de seguridad:")
for header, valor in headers_seguridad.items():
    print(f"  {header}: {valor}")

## Certificados Autofirmados para Local/Staging

> Para desarrollo local o entornos staging sin dominio público, puedes crear un certificado autofirmado. Los navegadores mostrarán una advertencia, pero funciona perfecto para pruebas.

In [ ]:
certificado_autofirmado = """
# Crear certificado autofirmado válido por 365 días
openssl req -x509 -nodes -days 365 -newkey rsa:2048 \\
    -keyout /etc/nginx/ssl/localhost.key \\
    -out /etc/nginx/ssl/localhost.crt \\
    -subj "/C=ES/ST=Madrid/L=Madrid/O=Dev/CN=localhost"
"""

print(certificado_autofirmado)

def peticion_ssl_local(url):
    ctx = ssl.create_default_context()
    ctx.check_hostname = False
    ctx.verify_mode = ssl.CERT_NONE
    print(f"Petición a {url} (ignorando SSL en desarrollo)")
    print("NUNCA uses esto en producción!")
    return "Solo para desarrollo local"

print("\nPetición SSL local:")
peticion_ssl_local("https://localhost/api/health")

## Diagnóstico de Problemas de Certificados

> Los problemas más comunes con SSL son: el dominio no apunta al servidor, el puerto 80 está bloqueado (Certbot necesita acceso HTTP para verificar), o nginx no está configurado correctamente.

In [ ]:
import socket
import ssl
import datetime

def diagnosticar_ssl(dominio):
    print(f"=== Diagnóstico SSL para: {dominio} ===\n")
    
    print("1. Resolución DNS:")
    try:
        ip = socket.gethostbyname(dominio)
        print(f"   {dominio} → {ip} OK")
    except socket.gaierror:
        print(f"   ERROR: No resuelve")
        return
    
    print("\n2. Puerto 80 (HTTP):")
    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    sock.settimeout(5)
    resultado = sock.connect_ex((dominio, 80))
    sock.close()
    if resultado == 0:
        print(f"   Puerto 80: ABIERTO")
    else:
        print(f"   Puerto 80: CERRADO - Necesario para Certbot")
    
    print("\n3. Certificado SSL:")
    print("   (verificación simulada en desarrollo)")
    print("   Vence: 2025-09-17 (45 días restantes)")
    
    print("\nComandos útiles:")
    cmds = [
        "certbot certificates",
        "certbot renew --dry-run",
        "openssl s_client -connect dominio:443",
        "nginx -t && systemctl reload nginx",
    ]
    for cmd in cmds:
        print(f"  $ {cmd}")

diagnosticar_ssl("ejemplo.com")

## Script de Setup SSL Completo

Automatizar la obtención y configuración de certificados SSL para múltiples dominios.

In [ ]:
def setup_ssl_completo(dominio, email_certbot):
    print(f"=== SETUP SSL PARA {dominio} ===\n")
    
    print("Paso 1: Verificando prerequisitos...")
    prerequisitos = [
        ("nginx activo", True),
        ("puerto 80 en UFW", True),
        ("puerto 443 en UFW", True),
        ("certbot instalado", True),
    ]
    for nombre, ok in prerequisitos:
        estado = "OK" if ok else "FALTA"
        print(f"  {nombre}: {estado}")
    
    print(f"\nPaso 2: Obteniendo certificado SSL...")
    cmd = f"certbot --nginx -d {dominio} -d www.{dominio} --non-interactive --agree-tos --email {email_certbot}"
    print(f"  Simulando: {cmd}")
    print(f"  Certificado: /etc/letsencrypt/live/{dominio}/")
    
    print(f"\nPaso 3: Verificando renovación automática...")
    print("  Timer systemd: snap.certbot.renew.timer - OK")
    
    print(f"\nPaso 4: Configurando headers de seguridad SSL...")
    headers = [
        'Strict-Transport-Security: max-age=31536000',
        'X-Content-Type-Options: nosniff',
        'X-Frame-Options: SAMEORIGIN',
    ]
    for h in headers:
        print(f"  + {h}")
    
    print(f"""
=== RESUMEN SSL ===
Dominio: {dominio}
Certificado: /etc/letsencrypt/live/{dominio}/fullchain.pem
Clave privada: /etc/letsencrypt/live/{dominio}/privkey.pem
Renovación: Automática (systemd timer)
HTTPS redirect: HTTP → HTTPS (301)
HSTS: Habilitado (1 año)

URLs activas:
  https://{dominio}
  https://www.{dominio}
""")
    return True

setup_ssl_completo("miempresa.com", "admin@miempresa.com")

## Tips y Mejores Prácticas

> HSTS con preload es permanente: una vez que un navegador lo recibe, forzará HTTPS para siempre. Asegúrate de que HTTPS funciona perfectamente antes de activar includeSubDomains y preload.

> Usa certbot renew --dry-run periódicamente para verificar que la renovación automática funcionará. Si falla, tendrás tiempo de arreglarlo antes de que venza el certificado.

> Let's Encrypt tiene un límite de 5 certificados por dominio por semana. Usa --staging durante el desarrollo para no desperdiciar cuota.

> Activa HTTP/2 en nginx con listen 443 ssl http2. HTTP/2 es significativamente más rápido para cargar múltiples recursos en paralelo.

> Si el DNS de tu dominio no apunta a tu servidor, Certbot fallará con "Connection refused". Verifica con: dig +short tudominio.com

## Errores Comunes

### Ejecutar certbot con el DNS apuntando a otra IP

¿Por qué ocurre?
- Certbot necesita acceder a tu servidor via HTTP para verificar que controlas el dominio. Si el DNS apunta a otro lugar, la verificación falla.

Solución
- Verifica con dig +short tudominio.com que la IP coincide con tu VPS. Espera la propagación del DNS (puede tardar hasta 48h).

### Agregar HSTS antes de verificar que HTTPS funciona correctamente

¿Por qué ocurre?
- HSTS le dice al navegador que nunca use HTTP. Si HTTPS tiene problemas, los usuarios no podrán acceder.

Solución
- Primero verifica que HTTPS funciona perfectamente. Empieza con max-age pequeño y auméntalo gradualmente.

### No renovar el certificado después de cambiar la IP del servidor

¿Por qué ocurre?
- Los archivos del certificado están en el VPS antiguo. El nuevo servidor no tiene los certificados.

Solución
- En el nuevo servidor, ejecuta certbot --nginx -d tudominio.com para obtener nuevos certificados.

### Usar certbot certonly sin configurar nginx para HTTPS después

¿Por qué ocurre?
- certonly solo obtiene el certificado pero no configura nginx. El sitio seguirá sirviendo HTTP.

Solución
- Usa certbot --nginx para que configure nginx automáticamente, o edita manualmente nginx.conf para agregar la configuración SSL.

### Guardar la clave privada del certificado en un repositorio git

¿Por qué ocurre?
- privkey.pem es la clave privada de tu certificado. Si se expone, cualquiera puede hacerse pasar por tu servidor.

Solución
- Los certificados están en /etc/letsencrypt/ con permisos restrictivos. Nunca los copies a tu repositorio.